# Notebook 4: Hyperparameter Tuning
**PURPOSE:** Find optimal parameters for top models using Optuna.
**NOTE:** This notebook takes 2-4 hours. Start it and come back.

In [6]:
import os
project_root = r"C:\Users\srich\OneDrive\Desktop\prediction-model"
os.chdir(project_root)
print(f"✅ Working directory: {os.getcwd()}")

✅ Working directory: C:\Users\srich\OneDrive\Desktop\prediction-model


In [7]:
import pandas as pd
import numpy as np
import pickle, json, time, warnings, subprocess
warnings.filterwarnings('ignore')

try:
    import optuna
except ImportError:
    subprocess.run(['pip', 'install', 'optuna', '--quiet'], check=True)
    import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.metrics import brier_score_loss
from sklearn.model_selection import TimeSeriesSplit
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier

print('Libraries loaded ✅')
print(f'Optuna version: {optuna.__version__}')

Libraries loaded ✅
Optuna version: 4.8.0


In [8]:
train_df = pd.read_csv('data/processed/training_data.csv')
test_df = pd.read_csv('data/processed/test_data.csv')
with open('model/feature_names.json') as f:
    feature_cols = json.load(f)

X_train = train_df[feature_cols].values
y_train = train_df['result'].values
X_test = test_df[feature_cols].values
y_test = test_df['result'].values

tscv = TimeSeriesSplit(n_splits=5)
print('Data loaded ✅')

Data loaded ✅


In [9]:
def xgb_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 800),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma': trial.suggest_float('gamma', 0, 2),
        'reg_alpha': trial.suggest_float('reg_alpha', 0, 2),
        'reg_lambda': trial.suggest_float('reg_lambda', 0, 2)
    }
    model = XGBClassifier(**params, eval_metric='logloss', random_state=42, n_jobs=-1, verbosity=0)
    scores = []
    for tr, val in tscv.split(X_train):
        model.fit(X_train[tr], y_train[tr])
        prob = model.predict_proba(X_train[val])[:, 1]
        scores.append(brier_score_loss(y_train[val], prob))
    return np.mean(scores)

def lgbm_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 800),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 20, 100),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0, 2),
        'reg_lambda': trial.suggest_float('reg_lambda', 0, 2)
    }
    model = LGBMClassifier(**params, random_state=42, n_jobs=-1, verbose=-1)
    scores = []
    for tr, val in tscv.split(X_train):
        model.fit(X_train[tr], y_train[tr])
        prob = model.predict_proba(X_train[val])[:, 1]
        scores.append(brier_score_loss(y_train[val], prob))
    return np.mean(scores)

def rf_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 5, 20),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'max_features': trial.suggest_float('max_features', 0.3, 1.0)
    }
    model = RandomForestClassifier(**params, random_state=42, n_jobs=-1)
    scores = []
    for tr, val in tscv.split(X_train):
        model.fit(X_train[tr], y_train[tr])
        prob = model.predict_proba(X_train[val])[:, 1]
        scores.append(brier_score_loss(y_train[val], prob))
    return np.mean(scores)

print('Objective functions defined ✅')

Objective functions defined ✅


## 4.2 Run Optuna Studies
**This takes 2-4 hours. Let it run.**

In [ ]:
best_params = {}
N_TRIALS = 100

objectives = {'XGBoost': xgb_objective, 'LightGBM': lgbm_objective} # Skipped RF Optuna

for model_name, objective in objectives.items():
    print(f'\n{"="*40}')
    print(f'Tuning: {model_name} ({N_TRIALS} trials)')
    print(f'{"="*40}')
    start = time.time()
    study = optuna.create_study(direction='minimize', study_name=model_name)
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)
    elapsed = time.time() - start
    best_params[model_name] = study.best_params
    print(f'\n✅ Done in {elapsed/60:.1f} min')
    print(f'Best Brier: {study.best_value:.4f}')
    print(f'Best params: {study.best_params}')

with open('model/best_params.json', 'r') as f:
    existing = json.load(f)
existing.update(best_params)
with open('model/best_params.json', 'w') as f:
    json.dump(existing, f, indent=2)
print('\n✅ Saved: model/best_params.json')


Tuning: XGBoost (100 trials)


  0%|          | 0/100 [00:00<?, ?it/s]

[W 2026-03-26 10:14:14,198] Trial 2 failed with parameters: {'n_estimators': 781, 'max_depth': 4, 'learning_rate': 0.07330977367390172, 'subsample': 0.6461469885024876, 'colsample_bytree': 0.7134846069395097, 'min_child_weight': 6, 'gamma': 0.3981372927349369, 'reg_alpha': 0.9106739849378125, 'reg_lambda': 0.44407687724589207} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "c:\Users\srich\anaconda3\Lib\site-packages\optuna\study\_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "C:\Users\srich\AppData\Local\Temp\ipykernel_22660\2268743209.py", line 16, in xgb_objective
    model.fit(X_train[tr], y_train[tr])
    ~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\srich\anaconda3\Lib\site-packages\xgboost\core.py", line 751, in inner_f
    return func(**kwargs)
  File "c:\Users\srich\anaconda3\Lib\site-packages\xgboost\sklearn.py", line 1806, in fit
    self._Booster = train(
                    ~~~~~^
      

KeyboardInterrupt: 


Tuning: XGBoost (100 trials)


KeyboardInterrupt: 

## 4.3 Train Final Tuned Models

In [ ]:
from sklearn.metrics import accuracy_score, brier_score_loss, roc_auc_score

with open('model/best_params.json') as f:
    best_params = json.load(f)

tuned_models = {}
tuned_results = []

model_classes = {
    'XGBoost': (XGBClassifier, {'eval_metric': 'logloss', 'random_state': 42, 'n_jobs': -1, 'verbosity': 0}),
    'LightGBM': (LGBMClassifier, {'random_state': 42, 'n_jobs': -1, 'verbose': -1}),
    'Random Forest': (RandomForestClassifier, {'random_state': 42, 'n_jobs': -1})
}

for name, (ModelClass, fixed_params) in model_classes.items():
    if name not in best_params: continue
    print(f'Training tuned {name}...')
    params = {**best_params[name], **fixed_params}
    model = ModelClass(**params)
    start = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - start
    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = model.predict(X_test)
    result = {
        'model': f'{name} (tuned)',
        'accuracy': accuracy_score(y_test, y_pred),
        'brier': brier_score_loss(y_test, y_prob),
        'roc_auc': roc_auc_score(y_test, y_prob),
        'train_time': train_time
    }
    tuned_results.append(result)
    tuned_models[name] = model
    safe = name.lower().replace(' ', '_')
    with open(f'model/tuned_{safe}.pkl', 'wb') as f:
        pickle.dump(model, f)
    print(f'  Brier: {result["brier"]:.4f}  Accuracy: {result["accuracy"]:.4f}')

tuned_df = pd.DataFrame(tuned_results).sort_values('brier')
print('\n=== TUNED MODEL RESULTS ===')
print(tuned_df.to_string(index=False))

Training tuned XGBoost...
  Brier: 0.1494  Accuracy: 0.7838
Training tuned LightGBM...
  Brier: 0.1492  Accuracy: 0.7830
Training tuned Random Forest...
  Brier: 0.1665  Accuracy: 0.7532

=== TUNED MODEL RESULTS ===
                model  accuracy    brier  roc_auc  train_time
     LightGBM (tuned)  0.783050 0.149157 0.864145    1.395579
      XGBoost (tuned)  0.783771 0.149423 0.863907    0.999227
Random Forest (tuned)  0.753220 0.166498 0.847362    4.089473
